In [37]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

In [38]:
from src.data_pull import save_model_results
import joblib

In [ ]:
pitcher_games = pd.read_csv("../data/processed/pitcher_games_features_base.csv")

pitcher_games["game_date"] = pd.to_datetime(pitcher_games["game_date"])

pitcher_games = pitcher_games.sort_values(["pitcher", "game_date"])

pitcher_games.shape

(5228, 84)

In [58]:
pitcher_games["baseline_k"] = (pitcher_games.groupby("pitcher")["strikeouts"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))

In [59]:
baseline_df = pitcher_games.dropna(subset=["baseline_k"]).copy()

In [60]:
cutoff = pd.Timestamp("2024-08-15")

train = baseline_df[baseline_df["game_date"] < cutoff]

test = baseline_df[baseline_df["game_date"] >= cutoff]

In [61]:
baseline_mae = mean_absolute_error(test["strikeouts"], test["baseline_k"])

baseline_rmse = np.sqrt(mean_squared_error(test["strikeouts"], test["baseline_k"]))

baseline_r2 = r2_score(test["strikeouts"], test["baseline_k"])

print(f"Baseline MAE: {baseline_mae:.3f}")
print(f"Baseline RMSE: {baseline_rmse:.3f}")
print(f"Baseline R²: {baseline_r2:.3f}")

Baseline MAE: 1.954
Baseline RMSE: 2.449
Baseline R²: 0.003


In [ ]:
drop_cols = [
    "game_pk", 
    "game_date", 
    "player_name", 
    "pitcher", 
    "strikeouts", 
    "baseline_k", 
    "home_team", 
    "away_team", 
    "opponent", 
    "pitches",
    "avg_velocity",
    "avg_spin",
    "avg_break_x",
    "avg_break_z",
    "csw",
    "whiffs",
    "called_strikes",
    "CSW%"
]

feature_cols = [col for col in pitcher_games if col not in drop_cols]

In [ ]:
pitch_mix_cols = [col for col in feature_cols if col.endswith("_usage")]

In [64]:
feature_cols = [col for col in feature_cols if col not in pitch_mix_cols]

In [65]:
X_train = train[feature_cols]
y_train = train["strikeouts"]

X_test = test[feature_cols]
y_test = test["strikeouts"]

In [66]:
ridge_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")), 
        ("scaler", StandardScaler()), 
        ("ridge", Ridge(alpha=1.0))
    ]
)

In [67]:
ridge_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](49,)","['rest_days','max_times_through_order','k_last3',..., 'rolling3_called_strikes','rolling5_called_strikes', 'season_called_strikes']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,49
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numerica

In [68]:
ridge_pred = ridge_pipeline.predict(X_test)

In [69]:
ridge_mae = mean_absolute_error(y_test, ridge_pred)

ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))

ridge_r2 = r2_score(y_test, ridge_pred)

print(f"Ridge MAE: {ridge_mae:.3f}")
print(f"Ridge RMSE: {ridge_rmse:.3f}")
print(f"Ridge R²: {ridge_r2:.3f}")

Ridge MAE: 1.756
Ridge RMSE: 2.218
Ridge R²: 0.182


In [70]:
results = pd.DataFrame( { "Model": [ "Rolling 5 Game Average", "Ridge Regression" ], "MAE": [ baseline_mae, ridge_mae ], "RMSE": [ baseline_rmse, ridge_rmse ], "R2": [ baseline_r2, ridge_r2 ] } )

results

,Model,MAE,RMSE,R2
0,Rolling 5 Game Average,1.953579,2.448653,0.002633
1,Ridge Regression,1.756359,2.217648,0.181939


In [71]:
save_model_results(results, 'Ridge_regression')

Saving as Ridge_regression_results.csv


In [72]:
joblib.dump(ridge_pipeline,"../models/ridge.pkl")

['../models/ridge.pkl']

In [73]:
len(feature_cols)

49

In [74]:
feature_cols

['rest_days',
 'max_times_through_order',
 'k_last3',
 'velo_last3',
 'spin_last3',
 'csw_last3',
 'pitches_last3',
 'velo_last6',
 'velo_trend',
 'k_std_last5',
 'pitches_last5',
 'whiff_last3',
 'csw_last5',
 'is_starter',
 'opp_runs_last14',
 'opp_k_rate_last14',
 'opp_bb_rate_last14',
 'opp_hr_rate_last14',
 'opp_iso_last14',
 'park_run_factor',
 'park_hr_factor',
 'last_avg_velocity',
 'rolling3_avg_velocity',
 'rolling5_avg_velocity',
 'season_avg_velocity',
 'last_avg_spin',
 'rolling3_avg_spin',
 'rolling5_avg_spin',
 'season_avg_spin',
 'last_avg_break_x',
 'rolling3_avg_break_x',
 'rolling5_avg_break_x',
 'season_avg_break_x',
 'last_avg_break_z',
 'rolling3_avg_break_z',
 'rolling5_avg_break_z',
 'season_avg_break_z',
 'last_csw',
 'rolling3_csw',
 'rolling5_csw',
 'season_csw',
 'last_whiffs',
 'rolling3_whiffs',
 'rolling5_whiffs',
 'season_whiffs',
 'last_called_strikes',
 'rolling3_called_strikes',
 'rolling5_called_strikes',
 'season_called_strikes']